# Allergy Intelligence Assistant
## Web Search + Medical MCP + Agentic RAG + Qdrant

### Main research question

**How can we understand allergy from early symptoms to mechanism, environmental effects, prevention, consultation, immunology research, and the latest medical treatment?**

This project follows the same MCP / context-engineering pattern as the reference notebook, but applies it to allergy and immunology.

There is **no hard-coded evidence snapshot**. Evidence is acquired at runtime from live sources, evaluated by agents, and stored in Qdrant.

```text
Allergy Research Question
        ↓
Biomedical Research Agent
        ↓
 ┌───────────────────────────────┐
 │ Tavily MCP                    │
 │ real-time web search/extract  │
 │                               │
 │ Medical MCP                   │
 │ PubMed / FDA / guidelines /   │
 │ medical databases             │
 └───────────────────────────────┘
        ↓
Source-quality evaluation
        ↓
Evidence Curator
        ↓
Qdrant MCP
(vector knowledge base)
        ↓
Allergy Agentic RAG
        ↓
Evidence retrieval + comparison
        ↓
Complete Allergy Education Story
        ↓
Prevention + Consultation Planner
```

> Educational research project only. It does not diagnose allergy, replace an allergist, or provide emergency medical care.


## Project Coverage

The knowledge base is designed to cover:

1. **Early symptoms** — allergic rhinitis, conjunctivitis, skin symptoms, respiratory symptoms, food reactions, and warning signs.
2. **Environmental effects** — pollen, mold, dust mites, pets, air pollution, humidity, weather, indoor environment, and seasonal exposure.
3. **Allergy mechanism** — sensitization, IgE, mast cells, basophils, histamine, Th2 immunity, cytokines, epithelial barrier responses, and immune tolerance.
4. **Latest immunology research** — emerging pathways, biomarkers, biologics, tolerance, microbiome research, and precision allergy approaches.
5. **Individual prevention** — trigger reduction, environmental control, symptom tracking, exposure planning, and evidence-based prevention.
6. **Common allergy knowledge** — what is well established, what is uncertain, and common misconceptions.
7. **Regular consultation** — when to see an allergist, what information to bring, testing, follow-up, and questions to discuss.
8. **Latest medical treatment** — antihistamines, corticosteroids, leukotriene approaches, epinephrine for anaphylaxis, allergen immunotherapy, oral/sublingual approaches, and biologic therapies when supported by current evidence.
9. **Clinical research** — recent trials, investigational therapies, and what remains unproven.

The agents must distinguish general education from diagnosis and personalized treatment.


# 1. Setup

This version is designed for **Windows + VS Code/Jupyter**.

Required `.env`:

```text
OPENAI_API_KEY=your_openai_key
TAVILY_API_KEY=your_tavily_key
```

Medical MCP does not require a separate API key in the configuration used here.

MCP workflows run in a normal Python subprocess using the same Python interpreter as the notebook. Jupyter's own event loop on Windows cannot spawn subprocesses, and an MCP server *is* a subprocess, so each workflow below is written as a small script and handed to `run_mcp`.

The next cell also settles two things once, so the workflows themselves stay readable:

1. **A certificate bundle for child processes.** On a network where a proxy or antivirus inspects TLS, HTTPS is re-signed with a root that Windows trusts but Python's `certifi` list does not. Every download then fails, the server dies before it can speak MCP, and the notebook reports only `Connection closed`. `make_ca_bundle.py` merges the two lists, and the cell also drops `SSL_CERT_DIR` and `SSLKEYLOGFILE`, which make a child process built against a different OpenSSL abort with `no OPENSSL_Applink`.
2. **UTF-8 decoding and live output.** The medical server prints emoji, which the default Windows codec cannot decode, and a research run takes many minutes. Output is streamed as it arrives instead of appearing all at once at the end.


In [1]:
import os
import subprocess
import sys
import tempfile
import textwrap
import threading
from pathlib import Path

from dotenv import load_dotenv

from make_ca_bundle import build as build_ca_bundle

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"

NOTEBOOK_DIR = Path.cwd()

# Anaconda exports these and they point a child process at a different OpenSSL
# build. A server that inherits them aborts with "no OPENSSL_Applink", which
# reaches the notebook only as the unhelpful "Connection closed".
UNSAFE_VARS = (
    "SSLKEYLOGFILE",
    "SSL_CERT_DIR",
    "__CONDA_OPENSSL_CERT_DIR_SET",
    "__CONDA_OPENSSL_CERT_FILE_SET",
)

# A proxy or antivirus that inspects HTTPS re-signs it with a root that Windows
# trusts but certifi does not, so npm, uvx and the embedding-model download all
# fail. This bundle merges certifi with the Windows store so both are trusted.
CA_BUNDLE = str(build_ca_bundle())


def build_mcp_env():
    """The environment every MCP subprocess inherits: cleaned, with working TLS.

    Building it once here means each script below can simply pass os.environ to
    its servers instead of repeating the certificate handling.
    """
    env = {key: value for key, value in os.environ.items() if key not in UNSAFE_VARS}

    env.update(
        {
            "SSL_CERT_FILE": CA_BUNDLE,
            "REQUESTS_CA_BUNDLE": CA_BUNDLE,
            "CURL_CA_BUNDLE": CA_BUNDLE,
            "UV_SYSTEM_CERTS": "1",              # uvx trusts the OS store
            "NODE_OPTIONS": "--use-system-ca",   # node trusts it too
            "PYTHONIOENCODING": "utf-8",
            "PYTHONUNBUFFERED": "1",             # stream output instead of buffering it
        }
    )

    return env


MCP_ENV = build_mcp_env()


def show(line):
    """Print a line from a server, dropping anything the console cannot render.

    The medical server prints emoji. Jupyter displays them, but a plain Windows
    console encodes in cp1252 and would raise UnicodeEncodeError instead.
    """
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    print(line.encode(encoding, "replace").decode(encoding, "replace"), end="")


def run_mcp(code_text, timeout=1800):
    """Run an MCP workflow in a subprocess using this notebook's Python.

    Output is streamed rather than captured, so a research run lasting many
    minutes shows progress instead of an empty cell. It is also decoded as
    UTF-8, since the servers do not print plain ASCII.
    """
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".py",
        delete=False,
        encoding="utf-8",
        dir=NOTEBOOK_DIR,
    ) as f:
        f.write(textwrap.dedent(code_text))
        script_path = f.name

    try:
        process = subprocess.Popen(
            [sys.executable, script_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=NOTEBOOK_DIR,
            env=MCP_ENV,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )

        watchdog = threading.Timer(timeout, process.kill)
        watchdog.start()

        try:
            with process:
                for line in process.stdout:
                    show(line)
        finally:
            watchdog.cancel()

        if process.returncode != 0:
            raise RuntimeError(
                f"MCP subprocess failed with return code {process.returncode}"
            )
    finally:
        try:
            os.remove(script_path)
        except OSError:
            pass


print("Python:", sys.executable)
print("Working directory:", NOTEBOOK_DIR)
print("OPENAI_API_KEY found:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY found:", bool(os.getenv("TAVILY_API_KEY")))


Wrote c:\Users\Sealion\Desktop\sos_2026 Study_new\sos agents-AI-main\6_mcp\memory\ca-bundle.pem (549 certificates from the Windows store)
Python: c:\Users\Sealion\Desktop\sos_2026 Study_new\sos agents-AI-main\.venv\Scripts\python.exe
Working directory: c:\Users\Sealion\Desktop\sos_2026 Study_new\sos agents-AI-main\6_mcp
OPENAI_API_KEY found: True
TAVILY_API_KEY found: True


# 2. Test the Live MCP Sources

We use three MCP servers:

### Tavily MCP
Real-time web search and content extraction.

### Medical MCP
A medical-information MCP integration with tools for medical literature, medical databases, FDA drug information, clinical guidelines, and related sources.

This one is launched through `medical_mcp_shim.py` rather than `npx`, because the published package has two bugs that stop it working as a stdio server:

- Its entry point has no `#!/usr/bin/env node` line. npm still builds a Windows launcher for it, but that launcher tries to execute the `.js` file directly instead of handing it to Node, so nothing runs. `npx` exits successfully having done nothing.
- It prints a startup banner with `console.log`. On a stdio server, stdout carries JSON-RPC and nothing else, so the client reads the banner, cannot parse it as a message, and drops the session.

Both surface identically, as `Error initializing MCP server: Connection closed`. The shim installs the package locally, runs it with `node` directly, and forwards back only the lines that are actually JSON-RPC.

The shim handles two more of the server's habits, this time while it is running. The server queries PubMed with no throttling, no API key and no retry, and NCBI allows anonymous callers only about three requests a second, so bursts come back as `Too Many Requests`. The shim spaces tool calls out to keep under that. The server also logs every failed lookup with a full Node stack trace, twenty lines for what is usually a transient problem, so those are collapsed to one line that keeps the message and the HTTP status.

Three of the server's ten tools are held back from the agents entirely:

- `search-medical-databases` can never succeed against ClinicalTrials.gov. It sends `query` and `limit`, where the v2 API expects `query.term` and `pageSize`, so that source always answers `400 Bad Request`. It also queries four sources at once, which is what provokes the PubMed rate limit.
- `search-google-scholar` and `search-medical-journals` scrape Google Scholar in a headless browser, the second one across five journals in parallel. Both are slow and routinely blocked as bot traffic.

None of these failures stop a run, because the server catches them and returns an empty list. That is exactly why they are worth removing: the agent cannot tell "this source is broken" from "nothing has been published", so it wastes turns and may record a gap that is not real.

### Qdrant MCP
Persistent vector knowledge base for Agentic RAG.

The first test only lists the available Tavily and Medical MCP tools. The first run is slow, since it downloads Tavily and installs Medical MCP; later runs start in seconds.


In [2]:
test_script = r'''
import asyncio
import gc
import os
import sys
from pathlib import Path

from agents.mcp import MCPServerStdio


def tavily_params():
    """Tavily runs straight from npx. It is a well-behaved stdio server."""
    return {
        "command": "npx.cmd" if sys.platform == "win32" else "npx",
        "args": ["-y", "tavily-mcp@latest"],
        "env": {**os.environ},
    }


def medical_params():
    """Medical MCP is launched through medical_mcp_shim.py.

    Running "npx -y medical-mcp" directly does not work on Windows, and the
    server also writes a banner to stdout, which is the JSON-RPC channel.
    The shim fixes both; its docstring explains the detail.
    """
    return {
        "command": sys.executable,
        "args": [str(Path("medical_mcp_shim.py").resolve())],
        "env": {**os.environ},
    }


# The tools the agents are actually given later. Sections 3 and 5 explain why
# the rest are held back: they either fail by construction or are far too slow.
TAVILY_TOOLS = ["tavily_search", "tavily_extract"]

MEDICAL_TOOLS = [
    "search-drugs",
    "get-drug-details",
    "search-drug-nomenclature",
    "get-health-statistics",
    "search-medical-literature",
    "get-article-details",
    "search-clinical-guidelines",
]


async def show_tools(label, params, timeout, enabled):
    """Connect to one server, list what it offers, then shut it down."""
    async with MCPServerStdio(
        params=params,
        client_session_timeout_seconds=timeout,
    ) as server:
        tools = await server.list_tools()

    print(f"\n{label} - {len(tools)} tools, {len(enabled)} used by the agents")
    for tool in tools:
        note = "" if tool.name in enabled else "   (held back)"
        print(f"  - {tool.name}{note}")


async def main():

    if not os.getenv("TAVILY_API_KEY"):
        raise RuntimeError("TAVILY_API_KEY is missing from .env")

    # First run downloads the server through npm, which is slow behind a proxy.
    await show_tools("Tavily MCP", tavily_params(), 300, TAVILY_TOOLS)

    # First run installs medical-mcp locally before it can answer.
    await show_tools("Medical MCP", medical_params(), 600, MEDICAL_TOOLS)

    await asyncio.sleep(0.5)


def run(workflow):
    """Run the workflow, then close the servers' pipes while the loop still runs.

    Those pipes are torn down by the garbage collector. If that happens after
    the loop has closed, each one raises "Event loop is closed" and prints a
    traceback under the cell, long after the real work succeeded.
    """
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(workflow())
        gc.collect()
        loop.run_until_complete(asyncio.sleep(0))
        loop.run_until_complete(loop.shutdown_asyncgens())
    finally:
        asyncio.set_event_loop(None)
        loop.close()


if __name__ == "__main__":
    run(main)
'''

run_mcp(test_script, timeout=1800)


Tavily MCP server running on stdio

Tavily MCP - 5 tools, 2 used by the agents
  - tavily_search
  - tavily_extract
  - tavily_crawl   (held back)
  - tavily_map   (held back)
  - tavily_research   (held back)
🚨 MEDICAL MCP SERVER - SAFETY NOTICE:
This server provides medical information for educational purposes only.
NEVER use this information as the sole basis for clinical decisions.
Always consult qualified healthcare professionals for patient care.
📊 DYNAMIC DATA SOURCE NOTICE:
This system queries live medical databases (FDA, WHO, PubMed, RxNorm)
NO hardcoded medical data is used - all information is retrieved dynamically
Data freshness depends on source database updates and API availability
Network connectivity required for all medical information retrieval
[medical-mcp] ✅ Medical MCP Server running on stdio

Medical MCP - 10 tools, 7 used by the agents
  - search-drugs
  - get-drug-details
  - get-health-statistics
  - search-medical-literature
  - get-article-details
  - search-

# 3. Build the Allergy Knowledge Base
## Tavily + Medical MCP → Research Agent → Qdrant

The **Allergy Research Agent** receives all three MCP servers.

It is responsible for planning multiple searches rather than receiving prewritten facts.

### Research strategy

The agent should use Tavily for broad and recent web discovery, and Medical MCP for focused medical literature, FDA/drug information, guidelines, and medical-database searches.

It should prefer primary or authoritative sources such as:

- NIH / NIAID / NIEHS / NHLBI
- FDA
- CDC when relevant
- PubMed-indexed peer-reviewed literature
- AAAAI / ACAAI
- ClinicalTrials.gov
- major peer-reviewed journals and academic medical centers

The agent should store only evidence that is useful, source-linked, and sufficiently reliable.


In [3]:
research_script = r'''
import asyncio
import gc
import os
import sys
from datetime import date
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

MODEL = "gpt-5.4-mini"

QUESTION = (
    "How can we understand allergy from early symptoms to mechanism, "
    "environmental effects, prevention, regular consultation, latest "
    "immunology research, and current medical treatment?"
)

# Tavily also ships tavily_research, an agentic deep-research tool that can run
# for many minutes. One call to it outlives the session timeout and fails the
# whole run, so the agent is shown only the two fast tools.
TAVILY_TOOLS = ["tavily_search", "tavily_extract"]

# The medical server offers ten tools. These seven are plain API calls to FDA,
# WHO, RxNorm and PubMed, and they behave. The three left out do not:
#   search-medical-databases  always fails against ClinicalTrials.gov, because
#                             it sends "query" and "limit" where the v2 API
#                             wants "query.term" and "pageSize". It also queries
#                             four sources at once, which triggers PubMed 429s.
#   search-google-scholar     scrapes Google Scholar in a headless browser,
#                             which is slow and routinely blocked as a bot.
#   search-medical-journals   the same scrape again, five journals in parallel.
MEDICAL_TOOLS = [
    "search-drugs",
    "get-drug-details",
    "search-drug-nomenclature",
    "get-health-statistics",
    "search-medical-literature",
    "get-article-details",
    "search-clinical-guidelines",
]


class FilteredMCPServer:
    # Wraps a server so the agent sees only the tools we name. The SDK used to
    # accept a tool_filter argument for this, but it was removed.

    def __init__(self, server, allowed_names):
        self._server = server
        self._allowed = set(allowed_names)

    async def list_tools(self):
        tools = await self._server.list_tools()
        return [tool for tool in tools if tool.name in self._allowed]

    async def call_tool(self, tool_name, tool_input):
        return await self._server.call_tool(tool_name, tool_input)

    def __getattr__(self, name):
        return getattr(self._server, name)


def tavily_params():
    """Tavily MCP: real-time web search and extraction."""
    return {
        "command": "npx.cmd" if sys.platform == "win32" else "npx",
        "args": ["-y", "tavily-mcp@latest"],
        "env": {**os.environ},
    }


def medical_params():
    """Medical MCP: literature, guidelines, drug and FDA information.

    Launched through medical_mcp_shim.py, which works around two packaging
    bugs in the published package. See the shim's docstring for the detail.
    """
    return {
        "command": sys.executable,
        "args": [str(Path("medical_mcp_shim.py").resolve())],
        "env": {**os.environ},
    }


def qdrant_params():
    """Qdrant MCP: the persistent vector knowledge base, stored on local disk."""
    qdrant_path = Path("memory/allergy_qdrant").resolve()
    embed_cache = Path("memory/fastembed").resolve()

    # Both must exist before the server starts, or it exits on the way up.
    qdrant_path.mkdir(parents=True, exist_ok=True)
    embed_cache.mkdir(parents=True, exist_ok=True)

    return {
        "command": "uvx",
        # --system-certs lets the PyPI download survive a proxy or antivirus
        # that re-signs HTTPS traffic.
        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],
        "env": {
            **os.environ,
            "QDRANT_LOCAL_PATH": str(qdrant_path),
            "COLLECTION_NAME": "allergy_evidence",
            "FASTEMBED_CACHE_PATH": str(embed_cache),
        },
    }


INSTRUCTIONS = """
You are an Allergy and Immunology Research Agent and Knowledge Curator.

Use Tavily MCP for real-time web discovery and content extraction.
Use Medical MCP for focused medical literature, clinical-guideline,
drug/FDA, and medical-database research.
Use Qdrant MCP to store the strongest evidence.

Your goal is NOT to give a quick answer.
Your goal is to build a high-quality allergy knowledge base for later Agentic RAG.

Research the following domains:

A. EARLY SYMPTOMS
- allergic rhinitis and conjunctivitis
- skin manifestations
- cough, wheeze and allergic asthma symptoms
- food-allergy reactions
- anaphylaxis warning signs
- differences between mild symptoms and medical emergencies

B. ENVIRONMENT
- pollen and seasonality
- dust mites
- mold and dampness
- pets
- cockroaches and indoor allergens
- humidity
- temperature and weather
- air pollution and smoke
- indoor air quality
- climate/environment interactions when supported

C. IMMUNOLOGY MECHANISM
- sensitization
- allergen-specific IgE
- mast cells and basophils
- histamine and inflammatory mediators
- Th2/type-2 immunity
- IL-4, IL-5, IL-13 when relevant
- epithelial barrier and alarmin pathways such as TSLP/IL-33/IL-25
  when supported by current evidence
- immune tolerance and desensitization

D. LATEST IMMUNOLOGY RESEARCH
- recent peer-reviewed allergy/immunology findings
- biomarkers
- biologics
- epithelial-barrier research
- immune tolerance
- microbiome research
- precision or personalized allergy approaches
Clearly mark early research as preliminary.

E. INDIVIDUAL PREVENTION
- evidence-based trigger reduction
- environmental control
- symptom/exposure tracking
- pollen/mold planning
- dust-mite and humidity management
- prevention strategies appropriate to the allergy type
Do not present one avoidance strategy as useful for every patient.

F. COMMON KNOWLEDGE AND MYTHS
- established allergy facts
- common misconceptions
- limits of allergy tests
- sensitization versus clinical allergy
- why history and testing must be interpreted together

G. REGULAR CONSULTATION
- when an allergist/immunologist evaluation is useful
- what information a person can track before a visit
- common diagnostic approaches
- follow-up considerations
- immunotherapy monitoring
- questions to discuss with a clinician

H. CURRENT AND EMERGING TREATMENT
- allergen avoidance/environmental management
- antihistamines
- topical/intranasal/inhaled corticosteroid approaches when relevant
- epinephrine and emergency management of anaphylaxis
- allergen immunotherapy
- subcutaneous and sublingual approaches
- oral immunotherapy when relevant
- biologics and targeted immunologic therapies
- recently approved therapies
- active or recent clinical trials
- what remains investigational

SOURCE PRIORITY:
1. FDA, NIH institutes, ClinicalTrials.gov and other government sources
2. PubMed-indexed peer-reviewed research
3. AAAAI / ACAAI and major professional guidance
4. major academic medical centers
5. other reputable sources only when necessary

WHEN A SEARCH COMES BACK EMPTY:
PubMed limits how fast anonymous callers may query it, so a medical search can
return nothing even on a well-covered topic. Read an empty result as "not
retrieved this time", never as "no evidence exists".
- do not repeat the same failing query more than once
- rephrase it, or approach the topic with a different tool, then move on
- prefer one well-aimed query over several rapid narrow ones
- list anything you could not verify in the evidence-gap report at the end

For every evidence chunk stored in Qdrant, preserve:
- topic
- allergy type
- population if relevant
- finding/recommendation
- evidence maturity
- publication/update date when available
- limitations
- source organization or journal
- title
- URL, PMID, DOI, or trial identifier when available

Store approximately 20-35 concise, non-duplicate evidence chunks.

Do not store unsupported claims.
Do not use promotional sources when authoritative sources are available.
Do not confuse association with causation.
Do not diagnose a person.
Do not recommend changing prescription treatment without a clinician.
Clearly label emergency symptoms and advise urgent/emergency care where appropriate.
"""

TASK = (
    "Build an evidence-grounded allergy knowledge base for this question:\n\n"
    f'"{QUESTION}"\n\n'
    f"Current date: {date.today().isoformat()}.\n\n"
    "Use multiple searches and medical database queries. "
    "Cover all eight research domains in the instructions. "
    "Evaluate evidence quality before storing it. "
    "Store the best evidence in Qdrant with source metadata. "
    "Finish with a short report describing what topics were stored "
    "and where important evidence gaps remain."
)


async def main():

    if not os.getenv("TAVILY_API_KEY"):
        raise RuntimeError("TAVILY_API_KEY is missing from .env")

    # All three servers stay open for the whole run, so the agent can move
    # between searching, verifying and storing without restarting anything.
    async with MCPServerStdio(
        params=tavily_params(),
        client_session_timeout_seconds=300,
    ) as tavily_server, MCPServerStdio(
        params=medical_params(),
        client_session_timeout_seconds=600,
    ) as medical_server, MCPServerStdio(
        params=qdrant_params(),
        client_session_timeout_seconds=600,
    ) as qdrant_server:

        agent = Agent(
            name="allergy_research_agent",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=[
                FilteredMCPServer(tavily_server, TAVILY_TOOLS),
                FilteredMCPServer(medical_server, MEDICAL_TOOLS),
                qdrant_server,
            ],
        )

        with trace("Allergy research and knowledge-base build"):
            result = await Runner.run(
                agent,
                TASK,
                max_turns=45,
            )

        print(result.final_output)

    await asyncio.sleep(0.5)


def run(workflow):
    """Run the workflow, then close the servers' pipes while the loop still runs.

    Those pipes are torn down by the garbage collector. If that happens after
    the loop has closed, each one raises "Event loop is closed" and prints a
    traceback under the cell, long after the real work succeeded.
    """
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(workflow())
        gc.collect()
        loop.run_until_complete(asyncio.sleep(0))
        loop.run_until_complete(loop.shutdown_asyncgens())
    finally:
        asyncio.set_event_loop(None)
        loop.close()


if __name__ == "__main__":
    run(main)
'''

run_mcp(research_script, timeout=3600)


Tavily MCP server running on stdio
🚨 MEDICAL MCP SERVER - SAFETY NOTICE:
This server provides medical information for educational purposes only.
NEVER use this information as the sole basis for clinical decisions.
Always consult qualified healthcare professionals for patient care.
📊 DYNAMIC DATA SOURCE NOTICE:
This system queries live medical databases (FDA, WHO, PubMed, RxNorm)
NO hardcoded medical data is used - all information is retrieved dynamically
Data freshness depends on source database updates and API availability
Network connectivity required for all medical information retrieval
[medical-mcp] ✅ Medical MCP Server running on stdio
C:\Users\Sealion\AppData\Local\uv\cache\archive-v0\9vbfeyHQHh_eaCpy\Lib\site-packages\fastmcp\server\auth\providers\bearer.py:6: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken
[08/28/26 12:23:29] INFO     Starti

# 4. Allergy Agentic RAG
## Qdrant → Retrieval Planning → Complete Evidence Story

The second agent receives **only Qdrant**.

This creates the retrieval half of RAG. It cannot simply perform another web search; it must reason over the evidence the first agent selected and stored.

The answer should connect allergy as a complete story:

```text
Exposure
   ↓
Sensitization / immune recognition
   ↓
IgE + type-2 immune response
   ↓
Mast-cell / basophil activation
   ↓
Symptoms
   ↓
Environment changes exposure
   ↓
Diagnosis + trigger identification
   ↓
Prevention + symptom control
   ↓
Immunotherapy / targeted treatment when appropriate
   ↓
Follow-up and monitoring
   ↓
New research may change future treatment
```


In [4]:
rag_script = r'''
import asyncio
import gc
import os
import sys
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

MODEL = "gpt-5.4-mini"

QUESTION = (
    "How can we understand allergy from early symptoms to mechanism, "
    "environmental effects, individual prevention, regular consultation, "
    "latest immunology research, and current treatment?"
)


def qdrant_params():
    """Point at the same collection the research agent wrote to."""
    return {
        "command": "uvx",
        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],
        "env": {
            **os.environ,
            "QDRANT_LOCAL_PATH": str(
                Path("memory/allergy_qdrant").resolve()
            ),
            "COLLECTION_NAME": "allergy_evidence",
            "FASTEMBED_CACHE_PATH": str(
                Path("memory/fastembed").resolve()
            ),
        },
    }


INSTRUCTIONS = """
You are an Allergy and Immunology Agentic RAG assistant.

Answer ONLY from evidence retrieved from the Qdrant allergy knowledge base.

Do not make one broad vector search and stop.
Plan retrieval by subtopic and search multiple times when useful.

Retrieve evidence for:
1. early symptoms and emergency warning signs
2. environmental influences
3. allergy immunology mechanism
4. recent immunology research
5. prevention and environmental control
6. common knowledge and misconceptions
7. consultation and follow-up
8. established and emerging treatments

Compare:
- established clinical practice
- recently approved treatment
- active clinical research
- preliminary mechanisms or hypotheses

For each important medical claim, preserve source information available
in the retrieved evidence.

Do not diagnose.
Do not create a patient-specific treatment plan.
Do not recommend stopping or changing prescribed medication.
For potential anaphylaxis or serious breathing difficulty, clearly state
that emergency evaluation/treatment is required.

Organize the final answer as a coherent story rather than disconnected facts.

Use these sections:

1. Allergy in one picture
2. Early symptoms: what often appears first
3. When symptoms become an emergency
4. How the allergic immune response works
5. Why the environment matters
6. Individual prevention and exposure reduction
7. Common allergy knowledge and misconceptions
8. Diagnosis and regular allergist consultation
9. Current established treatments
10. Allergen immunotherapy and immune tolerance
11. Biologics and targeted immunology
12. Latest allergy/immunology research
13. What is promising but still investigational
14. Practical questions to discuss with an allergist
15. Bottom line
"""

TASK = (
    f'Using only the Qdrant knowledge base, answer:\n\n"{QUESTION}"\n\n'
    "Create a complete evidence-grounded allergy education story. "
    "Clearly distinguish established care from recent or experimental research. "
    "Include source names and URLs/PMIDs/trial identifiers when they were stored."
)


async def main():

    # This agent is given Qdrant and nothing else. It cannot fall back on a web
    # search, so it has to reason over what the research agent chose to keep.
    async with MCPServerStdio(
        params=qdrant_params(),
        client_session_timeout_seconds=600,
    ) as qdrant_server:

        agent = Agent(
            name="allergy_rag_agent",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=[qdrant_server],
        )

        with trace("Allergy Agentic RAG retrieval"):
            result = await Runner.run(
                agent,
                TASK,
                max_turns=35,
            )

        print(result.final_output)

    await asyncio.sleep(0.5)


def run(workflow):
    """Run the workflow, then close the servers' pipes while the loop still runs.

    Those pipes are torn down by the garbage collector. If that happens after
    the loop has closed, each one raises "Event loop is closed" and prints a
    traceback under the cell, long after the real work succeeded.
    """
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(workflow())
        gc.collect()
        loop.run_until_complete(asyncio.sleep(0))
        loop.run_until_complete(loop.shutdown_asyncgens())
    finally:
        asyncio.set_event_loop(None)
        loop.close()


if __name__ == "__main__":
    run(main)
'''

run_mcp(rag_script, timeout=1800)


C:\Users\Sealion\AppData\Local\uv\cache\archive-v0\9vbfeyHQHh_eaCpy\Lib\site-packages\fastmcp\server\auth\providers\bearer.py:6: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken
[08/28/26 12:25:24] INFO     Starting MCP server                  server.py:981
                             'mcp-server-qdrant' with transport                
                             'stdio'                                           
## 1. Allergy in one picture

Allergy can be understood as a chain: **exposure → sensitization → immune activation → symptoms**. In established IgE-mediated allergy, an allergen is presented to the immune system, Th2/Tfh help promotes **class switching to allergen-specific IgE**, and IgE binds **FcεRI** on mast cells and basophils. On re-exposure, IgE is cross-linked and cells degranulate, releasing histamine and other mediators.  
**Source:

# 5. Live Medical Integration
## Qdrant + Medical MCP

The next stage combines the persistent RAG knowledge base with a **live medical integration**.

This is useful when a question needs both:

- previously curated allergy knowledge, and
- a fresh check of literature, guidelines, FDA information, or medical databases.

The integration agent should first retrieve background evidence from Qdrant, then use Medical MCP only when freshness or verification is needed.


In [5]:
integration_script = r'''
import asyncio
import gc
import os
import sys
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

MODEL = "gpt-5.4-mini"

QUESTION = (
    "What are the most important recent developments in allergy and immunology "
    "treatment, and which of them are established clinical options versus "
    "still investigational?"
)

# The same seven well-behaved medical tools used in section 3. The three left
# out either fail by construction or scrape Google Scholar in a browser; the
# comment in that cell explains each one.
MEDICAL_TOOLS = [
    "search-drugs",
    "get-drug-details",
    "search-drug-nomenclature",
    "get-health-statistics",
    "search-medical-literature",
    "get-article-details",
    "search-clinical-guidelines",
]


class FilteredMCPServer:
    # Wraps a server so the agent sees only the tools we name. The SDK used to
    # accept a tool_filter argument for this, but it was removed.

    def __init__(self, server, allowed_names):
        self._server = server
        self._allowed = set(allowed_names)

    async def list_tools(self):
        tools = await self._server.list_tools()
        return [tool for tool in tools if tool.name in self._allowed]

    async def call_tool(self, tool_name, tool_input):
        return await self._server.call_tool(tool_name, tool_input)

    def __getattr__(self, name):
        return getattr(self._server, name)


def medical_params():
    """Medical MCP, launched through the shim. See medical_mcp_shim.py."""
    return {
        "command": sys.executable,
        "args": [str(Path("medical_mcp_shim.py").resolve())],
        "env": {**os.environ},
    }


def qdrant_params():
    """The allergy knowledge base built in section 3."""
    return {
        "command": "uvx",
        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],
        "env": {
            **os.environ,
            "QDRANT_LOCAL_PATH": str(
                Path("memory/allergy_qdrant").resolve()
            ),
            "COLLECTION_NAME": "allergy_evidence",
            "FASTEMBED_CACHE_PATH": str(
                Path("memory/fastembed").resolve()
            ),
        },
    }


INSTRUCTIONS = """
You are an Allergy Evidence Integration Agent.

Use Qdrant first to retrieve the existing allergy knowledge base.
Use Medical MCP when current literature, FDA/drug information,
clinical guidance, or trial information needs fresh verification.

For treatment claims:
- distinguish FDA-approved indications from investigational uses
- distinguish guideline-supported care from early research
- give publication/update dates when available
- preserve source identifiers and links
- mention major evidence limitations

If a medical search returns nothing, treat it as "not retrieved this time"
rather than "no evidence exists". Rephrase once, or answer from the stored
evidence and say plainly that the live check could not be completed.

Do not diagnose a patient.
Do not recommend changing prescription treatment.
This is an evidence-research and education workflow.
"""


async def main():

    async with MCPServerStdio(
        params=medical_params(),
        client_session_timeout_seconds=600,
    ) as medical_server, MCPServerStdio(
        params=qdrant_params(),
        client_session_timeout_seconds=600,
    ) as qdrant_server:

        # Qdrant is listed first so the stored evidence is the agent's
        # starting point and the live lookup is the follow-up.
        agent = Agent(
            name="allergy_evidence_integration_agent",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=[
                qdrant_server,
                FilteredMCPServer(medical_server, MEDICAL_TOOLS),
            ],
        )

        with trace("Allergy medical integration"):
            result = await Runner.run(
                agent,
                QUESTION,
                max_turns=30,
            )

        print(result.final_output)

    await asyncio.sleep(0.5)


def run(workflow):
    """Run the workflow, then close the servers' pipes while the loop still runs.

    Those pipes are torn down by the garbage collector. If that happens after
    the loop has closed, each one raises "Event loop is closed" and prints a
    traceback under the cell, long after the real work succeeded.
    """
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(workflow())
        gc.collect()
        loop.run_until_complete(asyncio.sleep(0))
        loop.run_until_complete(loop.shutdown_asyncgens())
    finally:
        asyncio.set_event_loop(None)
        loop.close()


if __name__ == "__main__":
    run(main)
'''

run_mcp(integration_script, timeout=1800)


🚨 MEDICAL MCP SERVER - SAFETY NOTICE:
This server provides medical information for educational purposes only.
NEVER use this information as the sole basis for clinical decisions.
Always consult qualified healthcare professionals for patient care.
📊 DYNAMIC DATA SOURCE NOTICE:
This system queries live medical databases (FDA, WHO, PubMed, RxNorm)
NO hardcoded medical data is used - all information is retrieved dynamically
Data freshness depends on source database updates and API availability
Network connectivity required for all medical information retrieval
[medical-mcp] ✅ Medical MCP Server running on stdio
C:\Users\Sealion\AppData\Local\uv\cache\archive-v0\9vbfeyHQHh_eaCpy\Lib\site-packages\fastmcp\server\auth\providers\bearer.py:6: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken
[08/28/26 12:26:12] INFO     Starting MCP server                  serv

# 6. Individual Prevention + Consultation Planner

This final agent is intentionally **education-oriented**, not a diagnostic system.

It can turn a general, non-identifying scenario into:

- likely categories of triggers worth discussing,
- environmental observations to track,
- a symptom/exposure diary template,
- evidence-based prevention topics,
- questions for an allergist,
- red flags that should not wait for routine consultation.

It should never infer that a person has an allergy simply from symptoms.


In [6]:
planner_script = r'''
import asyncio
import gc
import os
import sys
from pathlib import Path

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

MODEL = "gpt-5.4-mini"

SCENARIO = """
Create a general prevention and consultation checklist for a person who
experiences recurring nasal/eye allergy-type symptoms that seem worse
in some indoor environments and during high-pollen seasons.

Do not diagnose the cause.
Explain what observations would be useful to track and what questions
could be discussed with an allergist.
"""


def qdrant_params():
    """The allergy knowledge base built in section 3."""
    return {
        "command": "uvx",
        "args": [
            "--system-certs",
            "mcp-server-qdrant",
        ],
        "env": {
            **os.environ,
            "QDRANT_LOCAL_PATH": str(
                Path("memory/allergy_qdrant").resolve()
            ),
            "COLLECTION_NAME": "allergy_evidence",
            "FASTEMBED_CACHE_PATH": str(
                Path("memory/fastembed").resolve()
            ),
        },
    }


INSTRUCTIONS = """
You are an Allergy Prevention and Consultation Planning Agent.

Use only the evidence retrieved from Qdrant.

Your purpose is to help organize observations and clinician discussions,
not to diagnose disease or prescribe treatment.

Create:
1. symptoms to track
2. timing and exposure information to track
3. indoor-environment observations
4. outdoor/pollen/weather observations
5. medication-response information to bring to a clinician
6. evidence-based exposure-reduction topics to discuss
7. questions for an allergist
8. reasons to arrange non-emergency follow-up
9. emergency warning signs that require immediate care

Make clear that allergy testing must be interpreted together with
clinical history and that sensitization does not automatically prove
that an exposure is causing symptoms.
"""


async def main():

    async with MCPServerStdio(
        params=qdrant_params(),
        client_session_timeout_seconds=600,
    ) as qdrant_server:

        agent = Agent(
            name="allergy_prevention_planner",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=[qdrant_server],
        )

        with trace("Allergy prevention and consultation planner"):
            result = await Runner.run(
                agent,
                SCENARIO,
                max_turns=25,
            )

        print(result.final_output)

    await asyncio.sleep(0.5)


def run(workflow):
    """Run the workflow, then close the servers' pipes while the loop still runs.

    Those pipes are torn down by the garbage collector. If that happens after
    the loop has closed, each one raises "Event loop is closed" and prints a
    traceback under the cell, long after the real work succeeded.
    """
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(
            asyncio.WindowsProactorEventLoopPolicy()
        )

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(workflow())
        gc.collect()
        loop.run_until_complete(asyncio.sleep(0))
        loop.run_until_complete(loop.shutdown_asyncgens())
    finally:
        asyncio.set_event_loop(None)
        loop.close()


if __name__ == "__main__":
    run(main)
'''

run_mcp(planner_script, timeout=1200)


C:\Users\Sealion\AppData\Local\uv\cache\archive-v0\9vbfeyHQHh_eaCpy\Lib\site-packages\fastmcp\server\auth\providers\bearer.py:6: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken
[08/28/26 12:26:55] INFO     Starting MCP server                  server.py:981
                             'mcp-server-qdrant' with transport                
                             'stdio'                                           
Here’s a general prevention and consultation checklist for recurring nasal/eye allergy-type symptoms that seem worse in some indoor environments and during high-pollen seasons.

This is **not a diagnosis**. Allergy testing, if done, must be interpreted together with the clinical history and symptom pattern; a **positive test shows sensitization, not automatically the cause of symptoms**.

## 1) Symptoms to track
Track the symptoms themselves

# 7. Complete Architecture

```text
                    ALLERGY QUESTION
                           │
                           ▼
              ┌────────────────────────┐
              │ Biomedical Research    │
              │ Agent                  │
              └───────────┬────────────┘
                          │
              ┌───────────┴───────────┐
              ▼                       ▼
       ┌─────────────┐          ┌─────────────┐
       │ Tavily MCP  │          │ Medical MCP │
       │ live web    │          │ PubMed/FDA/ │
       │ research    │          │ guidelines  │
       └──────┬──────┘          └──────┬──────┘
              └───────────┬────────────┘
                          ▼
              ┌────────────────────────┐
              │ Evidence Evaluation    │
              │ + Knowledge Curation   │
              └───────────┬────────────┘
                          ▼
                 ┌─────────────────┐
                 │   Qdrant MCP    │
                 │ allergy_evidence│
                 └────────┬────────┘
                          │
          ┌───────────────┼─────────────────┐
          ▼               ▼                 ▼
   ┌────────────┐  ┌──────────────┐  ┌──────────────┐
   │ Allergy RAG│  │ Live Medical │  │ Prevention + │
   │ Agent      │  │ Integration  │  │ Consultation │
   │            │  │ Agent        │  │ Planner      │
   └─────┬──────┘  └──────┬───────┘  └──────┬───────┘
         └─────────────────┴──────────────────┘
                           ▼
              ┌────────────────────────┐
              │ Evidence-grounded      │
              │ Allergy Intelligence   │
              └────────────────────────┘
```

### Why this is Agentic RAG

The system does more than retrieve documents.

The research agent decides **what to search**, **which source to trust**, **what evidence to keep**, and **how to structure the knowledge base**.

The RAG agent then decides **what subquestions require retrieval**, performs multiple semantic searches, compares evidence maturity, and produces a coherent synthesis.

The integration agent adds a fresh medical-source check when current evidence is important.


# 8. Suggested Research Questions

After the knowledge base is built, the architecture can be extended to questions such as:

- How can early allergic rhinitis symptoms be distinguished from common irritant symptoms?
- How do dust mites, mold, pollen, pets, humidity, and air pollution influence symptoms?
- What happens immunologically between first sensitization and a later allergic reaction?
- How are IgE, mast cells, basophils, Th2 cells, IL-4, IL-5, and IL-13 connected?
- What are researchers learning about epithelial barriers, TSLP, IL-33, immune tolerance, and the microbiome?
- Which environmental interventions have useful evidence and which are often overstated?
- What should someone track before a routine allergist consultation?
- How do allergy shots, sublingual immunotherapy, oral immunotherapy, and biologics differ?
- What allergy treatments have recently gained regulatory approval?
- What recent allergy clinical trials look promising but are not yet established care?
- What common allergy myths are inconsistent with current evidence?


# 9. Important Safety Boundary

This project is a **research and patient-education architecture**.

It should:

- explain evidence,
- organize questions,
- summarize environmental and immunologic concepts,
- support preparation for routine consultation,
- identify when evidence is preliminary.

It should not:

- diagnose allergy from chat symptoms,
- declare an allergen solely from a positive test,
- recommend stopping prescribed medication,
- substitute for an allergist,
- delay emergency care for anaphylaxis or serious breathing difficulty.
